# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will explore the dataset structure using its RecordSets and fields, each referenced by their `@id`.

In [ ]:
# Display record sets and their fields (using @id for all references)
record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets found in the Croissant schema.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs.id}")
        print(f"  name: {rs.name}")
        print(f"  description: {rs.description}")
        print(f"  Fields in this RecordSet:")
        for fld in rs.fields:
            print(f"    Field @id: {fld.id}")
            print(f"      name: {fld.name}")
            print(f"      dataType: {fld.data_type}")
        print('-' * 60)


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

Below, we extract all available record sets to DataFrames by using their `@id`.

In [ ]:
# Collect DataFrames per record set (referenced by @id)
record_sets = list(dataset.record_sets)
dataframes = {}
for rs in record_sets:
    print(f"Loading records for RecordSet @id: {rs.id}")
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"  Loaded {len(df)} records. Columns: {df.columns.tolist()}")

# If record_sets is not empty, show the first record set data
if record_sets:
    first_rs_id = record_sets[0].id
    print(f"First 5 records from RecordSet @id: {first_rs_id}")
    display(dataframes[first_rs_id].head())
else:
    print("No record sets available to display.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming distributions, or grouping data by key attributes.

**Note:** We reference all fields and columns by their `@id`. Please change `numeric_field_id` and `group_field_id` to match your dataset's available fields. You can refer to the field list from Section 2 or by printing DataFrame columns.

In [ ]:
# EDA example using field @id

# Please set these based on your dataset (see field @id from overview above)
record_set_id = list(dataframes.keys())[0] if dataframes else None
df = dataframes[record_set_id] if record_set_id else None

if df is not None and not df.empty:
    # Example: choose a numeric field and group field from your dataset by @id
    print("Available columns (usually @ids):", df.columns.tolist())
    
    # Attempt to auto-select first numeric column
    sample_numeric_cols = df.select_dtypes(include=['float', 'int']).columns
    if len(sample_numeric_cols) == 0:
        print("No numeric columns found for filtering. Please check the dataset fields and types.")
    else:
        numeric_field_id = sample_numeric_cols[0]
        print(f"Using numeric field @id: {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].mean() > 0 else 1
        # Filter records above threshold
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold} (mean):")
        print(filtered_df.head())
        # Normalize numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            (filtered_df[numeric_field_id].std() if filtered_df[numeric_field_id].std() > 0 else 1)
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by string/categorical field
        group_fields = df.select_dtypes(include='object').columns
        if len(group_fields) > 0:
            group_field_id = group_fields[0]
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
        else:
            print("No suitable group field found.")
else:
    print("No data available for EDA. Please check the dataset and record set fields.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We use field `@id`s as column names.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Histogram of a numeric field
if 'numeric_field_id' in locals() and df is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id], bins=30, kde=True)
    plt.title(f"Distribution of field @id: {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()
# Example: Scatter plot if there are at least two numeric fields
if df is not None and len(df.select_dtypes(include=['float', 'int']).columns) >= 2:
    cols = df.select_dtypes(include=['float', 'int']).columns[:2]
    plt.figure(figsize=(6, 6))
    sns.scatterplot(x=df[cols[0]], y=df[cols[1]])
    plt.xlabel(cols[0])
    plt.ylabel(cols[1])
    plt.title(f"Scatter plot of @id fields: {cols[0]} vs {cols[1]}")
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

This notebook demonstrated how to use the `mlcroissant` library to load, explore, and analyze a FAIR-compliant dataset (referencing every entity by its `@id`). You can extend this template to run downstream analysis or export subsets of the data for your research workflows.